# Completing Random Forest Analysis
Assumption that appropriate separation and PCA has been completed.

## SETUP

In [1]:
import subprocess, sys, os, importlib, re
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import rf_functions 
importlib.reload(rf_functions)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [2]:
# STEP 0 : LOAD THE ADSORPTION ENERGY DATA

def overall_rf(results_dir, transformed_data_dir, dataset):
    adsorption_energy_df = pd.read_excel('Adsorption_energies_additives.xlsx')

    # Initialize an empty list to store the results
    results = []
    
    files = [f for f in os.listdir(transformed_data_dir) if f.startswith('transformed')]

    for filename in files:
        filepath = os.path.join(transformed_data_dir, filename)

        # Load and process the data
        pca_data = rf_functions.load_pca_transformed_data(filepath)
        merged_data = rf_functions.adsorption_energy_for_dataset(pca_data, adsorption_energy_df)

        # Perform Random Forest and get mse and r2
        mae, mse, rmse, r2, adjusted_r2 = rf_functions.perform_random_forest(merged_data)

        # Extract the energy range and cumulative variance ratio from the filename
        energy_range_match = re.search(r'\[([-+]?\d*\.\d+|\d+),([-+]?\d*\.\d+|\d+)\]', filename)
        cum_variance_match = re.search(r'var([\d\.]+)', filename)

        if energy_range_match and cum_variance_match:
            energy_range = f"{energy_range_match.group(1)},{energy_range_match.group(2)}"
            cum_variance = cum_variance_match.group(1).rstrip('.')

            # Append the results to the list as a dictionary
            results.append({
                'dataset': dataset.replace('transformed_data_', ''),
                'energy_range': energy_range,
                'cum_variance': cum_variance,
                'n_pc': pca_data.shape[1] - 1,
                'MAE': mae,
                'MSE': mse,
                'RMSE': rmse,
                'R^2': r2,
                'Adjusted_R^2': adjusted_r2
            })

    # Convert the results list into a DataFrame
    results_df = pd.DataFrame(results)

    # Save results in overall results excel
    excel_name = os.path.join(results_dir, 'overall_results.xlsx')

    sheet_name = f'rf_results_{dataset}'

    # Save to Excel (append if file already exists)
    if os.path.exists(excel_name):
        # If Excel file does not exists, create a new Excel file
        with pd.ExcelWriter(excel_name, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
            results_df.to_excel(writer, sheet_name=sheet_name, index=False) 
    else:
        with pd.ExcelWriter(excel_name) as writer:
            results_df.to_excel(writer, sheet_name=sheet_name, index=False) 

    return results_df

In [3]:
# Create an empty DataFrame
overall_results = pd.DataFrame(columns=['dataset',
                                        'energy_range',
                                        'cum_variance',
                                        'n_pc',
                                        'MAE',
                                        'MSE',
                                        'RMSE',
                                        'R^2',
                                        'Adjusted_R^2'], dtype='float64')

overall_results

,dataset,energy_range,cum_variance,n_pc,MAE,MSE,RMSE,R^2,Adjusted_R^2


In [4]:
base_dir = os.path.dirname(os.getcwd())
results_dir = os.path.join(base_dir, 'RESULTS')

transformed_data_dirs = [f for f in os.listdir(results_dir) if f.startswith('transformed_data')]

for dir in transformed_data_dirs:
    print(f"Processing directory is {dir}")
    dir_path = os.path.join(results_dir, dir)
    cur_results = overall_rf(results_dir, dir_path, dir)
    overall_results = pd.concat([overall_results, cur_results], ignore_index=True).drop_duplicates()

overall_results

Processing directory is transformed_data_DOSCAR_Files


c:\Users\jespe\Desktop\[THESIS] Code\-ESPEJO-THESIS-DOS-ML-Model-\.venv\Lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Processing directory is transformed_data_DOSCAR_Files.xlsx


NotADirectoryError: [WinError 267] The directory name is invalid: 'c:\\Users\\jespe\\Desktop\\[THESIS] Code\\-ESPEJO-THESIS-DOS-ML-Model-\\RESULTS\\transformed_data_DOSCAR_Files.xlsx'

In [ ]:
## FOR PLOTTING y_test vs y_pred

adsorption_energy_df = pd.read_excel('Adsorption_energies_additives.xlsx')

def plot_rf_results(transformed_data_dir, filename, energy_range, variance):

    filepath = os.path.join(transformed_data_dir, filename)

    # Load and process the data
    pca_data = rf_functions.load_pca_transformed_data(filepath)
    merged_data = rf_functions.adsorption_energy_for_dataset(pca_data, adsorption_energy_df)

    # Prepare the data
    x = merged_data.drop(columns=['Molecule','Ads.'])
    y = merged_data['Ads.']

    # Split the data

    x_train, x_test, y_train, y_test = train_test_split(x, y,test_size=0.2, random_state=42)

    ## Train the Random Forest Model

    # Initialize and fit the model
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(x_train, y_train)

    ## Make predictions and evaluate the model

    # Predictions
    y_pred = rf_model.predict(x_test)

    # Extract sample indices from the first column of y_test (if it's a DataFrame)
    y_actual = y_test.values

    # Calculate axis limits for equal scaling
    min_val = min(y_actual.min(), y_pred.min()) - 0.2
    max_val = max(y_actual.max(), y_pred.max()) + 0.2

    # Plot actual vs predicted with y=x line
    plt.figure(figsize=(8, 8))
    plt.plot(y_actual, y_pred, 'o', markersize=6, label='Predictions')
    plt.plot([min_val, max_val], [min_val, max_val], 'r-', label='$y = x$', linewidth=2)  # y=x line

    plt.xlabel('Actual Adsorption Energy')
    plt.ylabel('Predicted Adsorption Energy')
    plt.title(f'Actual vs Predicted Adsorption Energy for {energy_range} and {variance * 100}% variance')
    plt.legend()
    plt.xlim(min_val, max_val)
    plt.ylim(min_val, max_val)
    plt.grid(True)
    plt.show()

## FOR DATASET 1

In [ ]:
transformed_data_dir = os.path.join(os.getcwd(),'transformed_data_DOSCAR_files_1st_Initial')

cur_results_set_1 = overall_rf(transformed_data_dir, 'Initial_set')
cur_results_set_1 

## FOR DATASET 1-5

In [ ]:
transformed_data_dir = os.path.join(os.getcwd(),'transformed_data_DOSCAR_files_2nd_Extended')

cur_results_set_5 = overall_rf(transformed_data_dir, 'Extended_set')
cur_results_set_5

In [ ]:
adsorption_energy_df = pd.read_excel('Adsorption_energies_additives.xlsx')

transformed_data_dir = os.path.join(os.getcwd(),'transformed_data_DOSCAR_files_2nd_Extended')
transformed_data_dir


In [ ]:
### FOR -5.00, 5.00 AND 90% VARIANCE

energy_range = '[-5.00,5.00]'
variance = 0.90

transformed_data_dir = os.path.join(os.getcwd(),'transformed_data_DOSCAR_files_2nd_Extended')
filename = f'transformed_data_{energy_range}_var{variance:.2f}.txt'

plot_rf_results(transformed_data_dir, filename, energy_range, variance)


In [ ]:
### FOR -10.00, 10.00 AND 90% VARIANCE

energy_range = '[-10.00,10.00]'
variance = 0.90

transformed_data_dir = os.path.join(os.getcwd(),'transformed_data_DOSCAR_files_2nd_Extended')
filename = f'transformed_data_{energy_range}_var{variance:.2f}.txt'

plot_rf_results(transformed_data_dir, filename, energy_range, variance)

In [ ]:
combined_df = pd.concat([cur_results_set_1, cur_results_set_5], ignore_index=True).drop_duplicates()
combined_df

# Save DataFrame to an Excel file
output_file = 'output_file.xlsx'
combined_df.to_excel(output_file, index=False)  # Set index=False to exclude the index column
print(f"DataFrame saved to {output_file}")

combined_df

In [ ]:

# Optimising hyper parameters and cross-validation